<center style="color:#1e3a8a;">
    <h1>YZ Bay Crane Analytics</h1>
    <h2>Predictive Maintenance for Crane LT Wheel Failure Reduction</h2>
</center>

<hr style="border:2px solid #2563eb">

<center style="color:#334155;">
    <h3>Backend Logic, Chart Creation &amp; Formulas Notebook</h3>
</center>

<br>

| **Module** | **Purpose** |
|:-----------|:------------|
| Dashboard | One-screen summary of all key numbers |
| Hardness Analysis | Rail hardness risk classification |
| Wheel Failure Analysis | Breakdown of wheel replacements |
| Failure Distribution | Failures by crane, quarter, day, severity |
| Rail Replacement Log | Record of rail pieces replaced |
| Predictions | Linear &amp; polynomial regression forecast |

---

## 🎯 Objective

To analyze and model the **YZ Bay Crane maintenance dataset** using predictive analytics techniques.

This notebook documents:
- How raw Excel data is loaded and cleaned
- The backend parsing logic used in `app.py` and `streamlit_app.py`
- How every chart is computed and rendered
- The formulas behind KPIs, risk classification, and failure predictions

## 📦 Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## ⚙️ Configuration Constants (Backend Logic)

These constants are defined in `app.py` / `streamlit_app.py` and drive data validation and parsing.

```python
REQUIRED_WHEEL_COLUMNS = {'Date', 'Crane', 'Equipment', 'Remarks'}
OPTIONAL_WHEEL_COLUMNS = {'S.no.', 'Job Description', 'Position'}
VALID_POSITIONS = {'SW', 'SE', 'NE', 'NW'}
VALID_CRANE_KEYWORDS = ['west', 'east']
DATA_PATH = 'data/LT Wheel replacement data.xlsx'
```

## 📥 Load Data from Excel

The workbook contains **3 sheets**:
1. `LT wheel replacement data` — every wheel replacement event
2. `Rail Hardness data` — HB hardness per rail section
3. `Rail Replacement data` — rail piece replacements and Thermit welding jobs

In [ ]:
DATA_PATH = 'data/LT Wheel replacement data.xlsx'
xls = pd.ExcelFile(DATA_PATH)
print('Sheets:', xls.sheet_names)

# Load raw sheets
df_wheels_raw = pd.read_excel(DATA_PATH, sheet_name='LT wheel replacement data')
df_hardness_raw = pd.read_excel(DATA_PATH, sheet_name='Rail Hardness data', header=None)
df_rail_raw = pd.read_excel(DATA_PATH, sheet_name='Rail Replacement data')

print('Wheel shape:', df_wheels_raw.shape)
print('Hardness shape:', df_hardness_raw.shape)
print('Rail shape:', df_rail_raw.shape)

## 🧹 Data Cleaning & Preprocessing

### Wheel Replacement Data

Backend logic from `parse_wheel_dataframe()`:
- Rename `S.no.` to `S_No`
- Parse `Date` with `dayfirst=True`
- Normalize crane names: any string containing **west** → `LT WEST`; **east** → `LT EAST`
- Extract wheel position (SW/SE/NE/NW) from `Equipment` or `Job Description` using regex word boundaries
- Fill missing Remarks with empty string

In [ ]:
VALID_POSITIONS = {'SW', 'SE', 'NE', 'NW'}

def normalize_crane(crane):
    if pd.isna(crane):
        return 'UNKNOWN'
    c = str(crane).lower()
    if 'west' in c:
        return 'LT WEST'
    if 'east' in c:
        return 'LT EAST'
    return str(crane).strip()

def extract_position(row):
    for col in ['Equipment', 'Job Description']:
        if col in row and pd.notna(row[col]):
            text = str(row[col]).upper()
            for pos in VALID_POSITIONS:
                if re.search(r'\\b' + pos + r'\\b', text):
                    return pos
    return 'Other'

def parse_wheel_dataframe(df_raw):
    df = df_raw.copy()
    df.columns = [c.strip() for c in df.columns]  # strip whitespace
    if 'S.no.' in df.columns:
        df = df.rename(columns={'S.no.': 'S_No'})
    for col in ['Date', 'Crane', 'Equipment', 'Remarks']:
        if col not in df.columns:
            df[col] = None
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce', dayfirst=True)
    df['Crane'] = df['Crane'].apply(normalize_crane)
    df['Position'] = df.apply(extract_position, axis=1)
    df['Position'] = df['Position'].apply(lambda x: x if x in VALID_POSITIONS else 'Other')
    df['Remarks'] = df['Remarks'].fillna('').astype(str)
    return df

df_wheels = parse_wheel_dataframe(df_wheels_raw)
df_wheels.head()

### Rail Hardness Data

Backend logic from `load_data_from_path()`:
- Layout: row 1 = section labels, row 2 = North HB, row 3 = South HB
- Columns 2–13 contain the 12 section values
- Convert values to numeric, coercing errors to NaN

In [ ]:
# Backend hardness parsing
sections = df_hardness_raw.iloc[1, 2:14].astype(str).tolist()
north = pd.to_numeric(df_hardness_raw.iloc[2, 2:14], errors='coerce').values
south = pd.to_numeric(df_hardness_raw.iloc[3, 2:14], errors='coerce').values

df_hardness = pd.DataFrame({'Section': sections, 'North': north, 'South': south})
print(df_hardness)

# Risk classification formula (from get_risk_class)
def get_risk_class(hb):
    if hb is None or pd.isna(hb):
        return 'Unknown'
    if hb > 400:
        return 'Critical'
    if hb > 350:
        return 'High'
    if hb > 300:
        return 'Medium'
    return 'Normal'

print('\nRisk thresholds: >400 Critical, >350 High, >300 Medium, else Normal')

### Rail Replacement Data

Backend logic from `parse_rail_replacement_dataframe()`:
- `Qty_Pieces`: extract first integer followed by `no.`/`no`/`nos.` from `Job Description`
- `Section`: extract `Column no. X to Y` or `Column no. X` from `Job Description`
- `Reason`: classify as `Thermit welding`, `Rail pieces replacement`, or `Maintenance`
- `Side`: defaults to `Both` for new-format logs

In [ ]:
def extract_rail_qty(job_description):
    if pd.isna(job_description):
        return 0
    match = re.search(r'(\\d+)\\s*no\.?', str(job_description), re.IGNORECASE)
    return int(match.group(1)) if match else 0

def extract_rail_section(job_description):
    if pd.isna(job_description):
        return 'Unknown'
    text = str(job_description)
    match = re.search(r'Column no\.\\s*(\\d+)\\s*to\\s*(\\d+)', text, re.IGNORECASE)
    if match:
        return f"Column {match.group(1)} to {match.group(2)}"
    match = re.search(r'Column no\.\\s*(\\d+)', text, re.IGNORECASE)
    if match:
        return f"Column {match.group(1)}"
    return 'Unknown'

def extract_rail_reason(equipment, job_description):
    text = ' '.join([str(equipment), str(job_description)]).lower()
    if 'thermit' in text or 'welding' in text:
        return 'Thermit welding'
    if 'replacement' in text:
        return 'Rail pieces replacement'
    return 'Maintenance'

df_rail = df_rail_raw.copy()
df_rail.columns = [c.strip() for c in df_rail.columns]
df_rail['Date'] = pd.to_datetime(df_rail['Date'], errors='coerce', dayfirst=True)
df_rail['Qty_Pieces'] = df_rail['Job Description'].apply(extract_rail_qty)
df_rail['Section'] = df_rail['Job Description'].apply(extract_rail_section)
df_rail['Reason'] = df_rail.apply(lambda r: extract_rail_reason(r.get('Equipment'), r.get('Job Description')), axis=1)
df_rail['Side'] = 'Both'
df_rail[['Date', 'Section', 'Side', 'Qty_Pieces', 'Reason']].head()

## 📊 Exploratory Data Analysis (EDA)

Summary statistics, missing values, and data types.

In [ ]:
print('=== Wheel Replacement Data ===')
print('Shape:', df_wheels.shape)
print('\nMissing values:')
print(df_wheels.isnull().sum())
print('\nData types:')
print(df_wheels.dtypes)
print('\nCrane counts:')
print(df_wheels['Crane'].value_counts())
print('\nPosition counts:')
print(df_wheels['Position'].value_counts())

print('\n=== Rail Hardness Data ===')
print(df_hardness.describe())

print('\n=== Rail Replacement Data ===')
print('Total pieces replaced:', df_rail['Qty_Pieces'].sum())
print('Total events:', len(df_rail))

## 🧮 KPI Formulas (Backend Logic)

The `/api/summary` endpoint in `app.py` computes:

| KPI | Formula | Code Location |
|:---|:---|:---|
| Total Replacements | `len(df_wheels)` | `app.py:1082` |
| West Crane Failures | count where Crane contains 'west' | `app.py:1083` |
| East Crane Failures | count where Crane contains 'East' | `app.py:1084` |
| Avg Hardness North | `mean(df_hardness['North'])` | `app.py:1095` |
| Avg Hardness South | `mean(df_hardness['South'])` | `app.py:1096` |
| Max Hardness | `max(all hardness values)` | `app.py:1097` |
| Above 300 HB North | `sum(h > 300 for h in north)` | `app.py:1098` |
| Above 300 HB South | `sum(h > 300 for h in south)` | `app.py:1099` |

Monthly trend is computed by grouping on `df['Date'].dt.to_period('M')`.

In [ ]:
df = df_wheels.dropna(subset=['Date']).copy()

total_replacements = len(df)
west_count = df[df['Crane'].astype(str).str.contains('west', case=False, na=False)].shape[0]
east_count = df[df['Crane'].astype(str).str.contains('East', case=False, na=False)].shape[0]

north_vals = df_hardness['North'].dropna().tolist()
south_vals = df_hardness['South'].dropna().tolist()
all_hardness = [float(v) for v in (north_vals + south_vals) if pd.notna(v)]

avg_north = float(np.mean(north_vals)) if north_vals else 0
avg_south = float(np.mean(south_vals)) if south_vals else 0
max_hb = float(max(all_hardness)) if all_hardness else 0
above_300_north = sum(1 for h in north_vals if h > 300)
above_300_south = sum(1 for h in south_vals if h > 300)

print(f'Total Replacements: {total_replacements}')
print(f'West Crane Failures: {west_count}')
print(f'East Crane Failures: {east_count}')
print(f'Avg Hardness North: {avg_north:.1f} HB')
print(f'Avg Hardness South: {avg_south:.1f} HB')
print(f'Max Hardness: {max_hb:.1f} HB')
print(f'Above 300 HB (North): {above_300_north}')
print(f'Above 300 HB (South): {above_300_south}')

## 📈 Dashboard Charts — How They Are Created

### 1. Monthly Wheel Replacements

Backend logic: `df['Month'] = df['Date'].dt.to_period('M')` then `groupby('Month').size()`.

In Streamlit this is rendered as a Plotly bar chart; here we use matplotlib.

In [ ]:
df['Month'] = df['Date'].dt.to_period('M').astype(str)
monthly = df.groupby('Month').size().reset_index(name='Count')

plt.figure(figsize=(10, 5))
sns.barplot(data=monthly, x='Month', y='Count', palette='Blues_d')
plt.title('Monthly Wheel Replacements')
plt.xlabel('Month')
plt.ylabel('Replacements')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Formula: count of records grouped by month')

### 2. Replacements by Crane

Backend logic: `df['Crane'].value_counts()` after normalizing names.

In [ ]:
crane_counts = df['Crane'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(crane_counts, labels=crane_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Replacements by Crane')
plt.axis('equal')
plt.show()

print('Formula: value_counts() on normalized Crane column')

### 3. Rail Hardness by Section (Line Chart)

Backend logic: plot North and South HB values per Section; add horizontal lines at 300 HB (threshold) and 400 HB (critical).

In [1]:
plt.figure(figsize=(12, 5))
plt.plot(df_hardness['Section'], df_hardness['North'], marker='o', label='North')
plt.plot(df_hardness['Section'], df_hardness['South'], marker='s', label='South')
plt.axhline(y=300, color='orange', linestyle='--', label='Threshold 300 HB')
plt.axhline(y=400, color='red', linestyle='--', label='Critical 400 HB')
plt.title('Rail Hardness by Section')
plt.xlabel('Section')
plt.ylabel('Hardness (HB)')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

### 4. Failures by Position

Backend logic: `df['Position'].value_counts()` after regex extraction.

In [ ]:
pos_counts = df['Position'].value_counts().reset_index()
pos_counts.columns = ['Position', 'Count']

plt.figure(figsize=(8, 5))
sns.barplot(data=pos_counts, x='Position', y='Count', palette='Pastel1')
plt.title('Failures by Position')
plt.xlabel('Position')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 🔬 Hardness Analysis

### Risk Classification Table

Each section × side gets a risk class and recommended action.

In [ ]:
risk_data = []
for _, row in df_hardness.iterrows():
    for side in ['North', 'South']:
        hb = row[side] if pd.notna(row[side]) else None
        risk = get_risk_class(hb)
        action = (
            'Immediate replacement' if risk == 'Critical'
            else 'Schedule replacement' if risk == 'High'
            else 'Monitor monthly' if risk == 'Medium'
            else 'Normal operation'
        )
        risk_data.append({
            'Section': row['Section'],
            'Side': side,
            'Hardness (HB)': hb,
            'Risk': risk,
            'Recommended Action': action
        })

risk_df = pd.DataFrame(risk_data)
print(risk_df.to_string(index=False))

print('\nRisk counts:')
print(risk_df['Risk'].value_counts())

### North vs South Hardness per Section (Grouped Bar)

In [ ]:
x = np.arange(len(df_hardness))
width = 0.35

plt.figure(figsize=(12, 5))
plt.bar(x - width/2, df_hardness['North'], width, label='North', color='#2563eb')
plt.bar(x + width/2, df_hardness['South'], width, label='South', color='#ec4899')
plt.axhline(y=300, color='orange', linestyle='--', label='Normal (300)')
plt.axhline(y=350, color='red', linestyle='--', label='High (350)')
plt.axhline(y=400, color='darkred', linestyle='--', label='Critical (400)')
plt.xticks(x, df_hardness['Section'], rotation=45)
plt.title('North vs South Hardness per Section')
plt.ylabel('Hardness (HB)')
plt.legend()
plt.tight_layout()
plt.show()

### Risk Distribution Pie Chart

In [ ]:
risk_counts = risk_df['Risk'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(risk_counts, labels=risk_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Risk Distribution')
plt.axis('equal')
plt.show()

## ⚙️ Wheel Failure Analysis

Backend logic from `/api/wheel-failure-analysis`:
- Top 10 equipment types by frequency
- Position counts
- Top 8 remarks
- Monthly total counts

In [ ]:
equipment_counts = df['Equipment'].value_counts().head(10)
position_counts = df['Position'].value_counts()
job_counts = df['Remarks'].value_counts().head(8)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Equipment failures
equipment_counts.plot(kind='barh', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Failures by Equipment Type')
axes[0,0].invert_yaxis()

# Position failures
position_counts.plot(kind='pie', ax=axes[0,1], autopct='%1.1f%%')
axes[0,1].set_title('Failures by Position')
axes[0,1].set_ylabel('')

# Remarks
job_counts.plot(kind='bar', ax=axes[1,0], color='coral')
axes[1,0].set_title('Failures by Remark Type')
axes[1,0].tick_params(axis='x', rotation=45)

# Monthly trend
monthly.plot(x='Month', y='Count', ax=axes[1,1], marker='o', color='green')
axes[1,1].set_title('Monthly Trend')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 📈 Failure Distribution

Backend logic from `/api/failure-distribution`:
- By Crane: `df['Crane'].value_counts()`
- By Quarter: `df['Date'].dt.to_period('Q')` then `groupby().size()`
- By Day of Week: `df['Date'].dt.day_name()` then `value_counts()`
- By Severity: keyword matching on `Remarks`

In [ ]:
df['Quarter'] = df['Date'].dt.to_period('Q').astype(str)
df['DayOfWeek'] = df['Date'].dt.day_name()

severity = {'Critical': 0, 'High': 0, 'Medium': 0, 'Low': 0}
for remark in df['Remarks'].astype(str):
    r = remark.lower()
    if any(x in r for x in ['crack', 'hot axle', 'bearing failure', 'breakdown']):
        severity['Critical'] += 1
    elif any(x in r for x in ['damaged', 'worn out', 'broken']):
        severity['High'] += 1
    elif any(x in r for x in ['wear', 'limit', 'reduced']):
        severity['Medium'] += 1
    else:
        severity['Low'] += 1

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# By Crane
df['Crane'].value_counts().plot(kind='pie', ax=axes[0,0], autopct='%1.1f%%')
axes[0,0].set_title('By Crane')
axes[0,0].set_ylabel('')

# By Quarter
quarterly = df.groupby('Quarter').size()
quarterly.plot(kind='bar', ax=axes[0,1], color='purple')
axes[0,1].set_title('Quarterly Trend')
axes[0,1].tick_params(axis='x', rotation=45)

# By Day of Week
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = df['DayOfWeek'].value_counts().reindex(day_order, fill_value=0)
dow.plot(kind='bar', ax=axes[1,0], color='teal')
axes[1,0].set_title('By Day of Week')
axes[1,0].tick_params(axis='x', rotation=45)

# By Severity
pd.Series(severity).plot(kind='bar', ax=axes[1,1], color='crimson')
axes[1,1].set_title('By Severity')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('Severity counts:', severity)

## 🛤️ Rail Replacement Log

Backend logic from `/api/rail-replacement`:
- `total_pieces = df['Qty_Pieces'].sum()`
- `total_events = len(df)`

In [ ]:
total_pieces = int(df_rail['Qty_Pieces'].sum())
total_events = len(df_rail)

print(f'Total Replacement Events: {total_events}')
print(f'Total Rail Pieces Replaced: {total_pieces}')

# By Reason
reason_counts = df_rail['Reason'].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(reason_counts, labels=reason_counts.index, autopct='%1.1f%%')
plt.title('Rail Replacement by Reason')
plt.axis('equal')
plt.show()

# Pieces by Section
section_qty = df_rail.groupby('Section')['Qty_Pieces'].sum().reset_index()
plt.figure(figsize=(10, 5))
sns.barplot(data=section_qty, x='Section', y='Qty_Pieces', palette='Reds_d')
plt.title('Rail Pieces by Section')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 🔮 Predictions — Linear & Polynomial Regression

Backend logic from `/api/predict/<months_ahead>` in `app.py` (lines 1266–1336).

### Step 1: Build Time Series
```python
df['Days'] = (df['Date'] - df['Date'].min()).dt.days
time_series = df.groupby('Days').size().cumsum().reset_index()
time_series.columns = ['Days', 'Cumulative_Failures']
```

### Step 2: Fit Models
- **Linear Regression**: `y = β₀ + β₁·days`
- **Polynomial Regression (degree 2)**: `y = β₀ + β₁·days + β₂·days²`

### Step 3: Hardness Risk Adjustment
```text
Hardness Risk Factor = max(0, (Average HB - 300) / 100)
Adjusted Prediction  = Raw Prediction × (1 + Hardness Risk Factor)
```

### Step 4: Recommendations
| Avg HB | Recommendation |
|:---|:---|
| > 400 | URGENT: Critical hardness — immediate rail replacement |
| > 350 | HIGH PRIORITY: Schedule comprehensive rail replacement |
| > 300 | MODERATE: Monitor hardness monthly, plan targeted replacement |
| ≤ 300 | NORMAL: Continue standard inspection intervals |

In [ ]:
months_ahead = 12

df['Days'] = (df['Date'] - df['Date'].min()).dt.days
time_series = df.groupby('Days').size().cumsum().reset_index()
time_series.columns = ['Days', 'Cumulative_Failures']

X = time_series['Days'].values.reshape(-1, 1)
y_vals = time_series['Cumulative_Failures'].values

lin_model = LinearRegression()
lin_model.fit(X, y_vals)

poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)
poly_model = LinearRegression()
poly_model.fit(X_poly, y_vals)

last_day = int(time_series['Days'].max())
future_days = [last_day + (30 * i) for i in range(1, months_ahead + 1)]

future_linear = lin_model.predict(np.array(future_days).reshape(-1, 1))
future_poly = poly_model.predict(poly.transform(np.array(future_days).reshape(-1, 1)))

all_hb = [float(v) for v in list(df_hardness['North']) + list(df_hardness['South']) if pd.notna(v)]
avg_hb = float(np.mean(all_hb)) if all_hb else 300
hardness_risk = max(0, (avg_hb - 300) / 100)

adjusted_linear = future_linear * (1 + hardness_risk)
adjusted_poly = np.maximum.accumulate(future_poly * (1 + hardness_risk))

current_failures = int(y_vals[-1])
pred_3mo = int(adjusted_poly[2] - current_failures) if months_ahead >= 3 else 0
pred_6mo = int(adjusted_poly[5] - current_failures) if months_ahead >= 6 else 0
monthly_rate = round((adjusted_poly[-1] - current_failures) / months_ahead, 1)

print(f'Current Total Failures: {current_failures}')
print(f'Next 3 Months (predicted): {pred_3mo}')
print(f'Next 6 Months (predicted): {pred_6mo}')
print(f'Avg Monthly Rate: {monthly_rate}')
print(f'Avg Hardness: {avg_hb:.1f} HB')
print(f'Hardness Risk Factor: {hardness_risk:.2f}')

if avg_hb > 400:
    print('Recommendation: URGENT: Critical hardness levels - immediate rail replacement required')
elif avg_hb > 350:
    print('Recommendation: HIGH PRIORITY: Schedule comprehensive rail replacement program')
elif avg_hb > 300:
    print('Recommendation: MODERATE: Monitor hardness monthly and plan targeted replacement')
else:
    print('Recommendation: NORMAL: Continue standard inspection intervals')

In [ ]:
future_months = [f'Month +{i+1}' for i in range(months_ahead)]
pred_df = pd.DataFrame({
    'Month': future_months,
    'Linear Forecast': [round(float(adjusted_linear[i] - current_failures), 1) for i in range(months_ahead)],
    'Polynomial Forecast': [round(float(adjusted_poly[i] - current_failures), 1) for i in range(months_ahead)]
})

plt.figure(figsize=(12, 5))
plt.plot(pred_df['Month'], pred_df['Linear Forecast'], marker='o', label='Linear', linestyle='--')
plt.plot(pred_df['Month'], pred_df['Polynomial Forecast'], marker='s', label='Polynomial')
plt.title('Cumulative Failure Forecast')
plt.xlabel('Future Month')
plt.ylabel('Additional Failures')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

print(pred_df.to_string(index=False))

## 📝 Complete Formula Reference

### Data Parsing
| Step | Formula / Rule |
|:---|:---|
| Crane normalization | `west → LT WEST`, `east → LT EAST` |
| Position extraction | regex `\b(SW\|SE\|NE\|NW)\b` on Equipment / Job Description |
| Rail qty extraction | regex `(\d+)\s*no\.?` on Job Description |
| Rail section extraction | regex `Column no\.\s*(\d+)\s*to\s*(\d+)` or `Column no\.\s*(\d+)` |

### Risk Classification
```text
HB > 400  → Critical
HB > 350  → High
HB > 300  → Medium
otherwise → Normal
```

### Predictive Models
```text
Linear:      y = β₀ + β₁·x
Polynomial:  y = β₀ + β₁·x + β₂·x²
Hardness Risk Factor = max(0, (avg_HB - 300) / 100)
Adjusted Prediction  = Raw Prediction × (1 + Hardness Risk Factor)
Monthly Failure Rate = (poly_final - current_failures) / months_ahead
```

### Severity Classification (Remarks keyword matching)
```text
Critical: crack, hot axle, bearing failure, breakdown
High:     damaged, worn out, broken
Medium:   wear, limit, reduced
Low:      everything else
```

## 🖥️ Backend Architecture Notes

### Files and Responsibilities
| File | Responsibility |
|:---|:---|
| `app.py` | Flask backend; REST API endpoints (`/api/*`), auth, upload, version history |
| `streamlit_app.py` | Standalone Streamlit UI; same analytics logic with Plotly charts |
| `data/crane_analytics.db` | SQLite database with `users` and `excel_versions` tables |
| `data/LT Wheel replacement data.xlsx` | Currently active Excel workbook |
| `data/version_N.xlsx` | Snapshots of every uploaded workbook |

### Key API Endpoints
| Endpoint | Returns |
|:---|:---|
| `GET /api/summary` | KPIs, monthly/yearly trends |
| `GET /api/hardness-data` | North/South HB arrays and thresholds |
| `GET /api/hardness-heatmap` | Risk classification per section × side |
| `GET /api/hardness-correlation` | Risk zones with recommended actions |
| `GET /api/wheel-failure-analysis` | Equipment, position, remark, monthly breakdown |
| `GET /api/failure-distribution` | Crane, quarter, day-of-week, severity breakdown |
| `GET /api/rail-replacement` | Parsed rail replacement records |
| `GET /api/predict/<months>` | Linear & polynomial forecast |
| `GET /api/scatter-hardness-failures` | Simulated hardness vs failure correlation |

### Authentication
- Passwords hashed with `bcrypt`
- Default admin: `admin` / `admin123`
- Two roles: `user` and `admin`

### Versioning
Every upload is saved as `version_{id}.xlsx` and tracked in `excel_versions`. Any previous version can be re-activated, replacing the active `LT Wheel replacement data.xlsx`.

## ✅ Conclusion

This notebook replicates and documents the complete backend logic of the YZ Bay Crane Analytics application:

1. **Data is loaded** from a 3-sheet Excel workbook.
2. **Data is cleaned** automatically — crane names normalized, positions extracted via regex, dates parsed.
3. **KPIs and charts** are computed fresh from the cleaned data every time.
4. **Risk classification** uses simple HB thresholds (300/350/400).
5. **Predictions** combine linear/polynomial regression with a hardness-risk multiplier.
6. **Everything is versioned** and stored securely with bcrypt-hashed passwords.

The same logic lives in `app.py` (Flask API) and `streamlit_app.py` (interactive UI).